# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# Check the Python interpreter and installed package versions.
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)

# Check the libraries required by the lab.
packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-openai",
    "chromadb",
    "python-dotenv",
    "beautifulsoup4"
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")
        

Python: 3.12.14
Interpreter: /Users/caiomaia/ironhack-workspace/.venv/langchain-v0.2.x/bin/python
langchain: 0.2.17
langchain-core: 0.2.43
langchain-community: 0.2.19
langchain-openai: 0.1.25
chromadb: NOT INSTALLED
python-dotenv: 1.2.4
beautifulsoup4: NOT INSTALLED


In [2]:
# Install only the two missing libraries.
# Keep the LangChain packages at their existing compatible versions.
%pip install "chromadb==1.5.9" "beautifulsoup4==4.15.0"

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.9.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached websocket_client-1.9.2-py3-none-any.whl.metadata (8.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached protobuf-7.36.2-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.5.0-py3-none-any.whl.metada

In [3]:
# Import the vector database integration used in this lab.
from langchain_community.vectorstores import Chroma

# Import the tools used to load and split documents.
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter

# Import the components used to build retrieval and agent workflows.
from langchain.chains import RetrievalQA
from langchain.agents import AgentType, Tool, initialize_agent

print("All required LangChain components imported successfully.")

USER_AGENT environment variable not set, consider setting it to identify your requests.


All required LangChain components imported successfully.


<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [4]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [6]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

In [9]:
# Import the chat-model wrapper.
from langchain_openai import ChatOpenAI

# Initialize the model with an explicit, supported model name.
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Verify the configured model.
print("Model:", llm.model_name)

Model: gpt-4.1-mini


<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [7]:
doc_path =  "./datasets/state_of_the_union.txt"

In [8]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [10]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [11]:
from langchain_community.document_loaders import WebBaseLoader

In [12]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [13]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [14]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [15]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [16]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/var/folders/1g/3xm4lnvn64z02xj5s9vlj7hc0000gn/T/ipykernel_55972/1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [17]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to find specific information about what President Biden said regarding Ketanji Brown Jackson in the most recent State of the Union address. I will use the State of Union QA System to get an accurate answer.

Action: State of Union QA System
Action Input: What did President Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, President Biden said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to the United States Supreme Court four days prior. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.
Thought:I now know the final answer  
Final Answer: In the State of the Union address, President Biden said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to the United States Supreme Court four days prior. He described her as one of the nation's top legal minds who will continue Justice Stephen Bre

{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, President Biden said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to the United States Supreme Court four days prior. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence."}

In [18]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: The question is about the advantages of using ruff over flake8, both of which are Python linters. To provide a detailed and accurate answer, I should query the Ruff QA System to get specific information about why one might prefer ruff over flake8.

Action: Ruff QA System
Action Input: Why should someone use ruff over flake8?

Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance and Speed**: Ruff is implemented in Rust, making it significantly faster than Flake8, which is Python-based. This speed advantage can lead to quicker linting feedback, especially on large codebases.

2. **Broader Rule Coverage**: Ruff implements over 900 rules, including all Flake8 rules plus many from popular Flake8 plugins, effectively consolidating multiple tools into one. This reduces the need to manage multiple plugins separately.

3. **Native Support Without Plugins**: Unlike Flake8, which relies heavily on third-party plugins for extended functionality,

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 because Ruff is significantly faster due to being implemented in Rust, offers broader rule coverage including many Flake8 plugins natively, supports automatic fixing of lint issues, works well alongside or can replace tools like Black and isort, simplifies configuration by reducing dependency on plugins, supports modern Python versions, and does not require installing Rust separately despite being written in it. Overall, Ruff provides a faster, more comprehensive, and more integrated linting experience compared to Flake8.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [19]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [20]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [21]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to find specific information about what President Biden said regarding Ketanji Brown Jackson in the most recent State of the Union address. I will use the State of Union QA System to get an accurate answer.

Action: State of Union QA System
Action Input: What did President Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, President Biden said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to the United States Supreme Court four days prior. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, President Biden said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to the United States Supreme Court four days prior. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence."}

In [22]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: The question is about the advantages of using ruff over flake8, both of which are Python linters. To provide a detailed and accurate answer, I should query the Ruff QA System to get specific information about why one might prefer ruff over flake8.

Action: Ruff QA System
Action Input: Why should someone use ruff over flake8?

Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance and Speed**: Ruff is implemented in Rust, making it significantly faster than Flake8, which is Python-based. This speed advantage can lead to quicker linting feedback, especially on large codebases.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including all Flake8 rules (F, and many E and W rules) plus re-implementations of popular Flake8 plugins natively. This means you get a broader set of linting checks out of the box without needing to install multiple plugins.

3. **No Need for Plugins**: Unlike Flake8, which relies heavily on third-par

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for several reasons:\n\n1. **Performance and Speed**: Ruff is implemented in Rust, making it significantly faster than Flake8, which is Python-based. This speed advantage can lead to quicker linting feedback, especially on large codebases.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including all Flake8 rules (F, and many E and W rules) plus re-implementations of popular Flake8 plugins natively. This means you get a broader set of linting checks out of the box without needing to install multiple plugins.\n\n3. **No Need for Plugins**: Unlike Flake8, which relies heavily on third-party plugins for extended functionality, Ruff includes many of these rules natively. This reduces complexity in configuration and dependency management.\n\n4. **Automatic Fixing**: Ruff can automatically fix many lint violations, whereas Flake8 is primarily a reporting tool and does not provide

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [23]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [25]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [26]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Question: What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?
Thought: I need to find out what tool ruff uses to run over Jupyter Notebooks first. Then I need to check if the president mentioned that tool in the state of the union address.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation: Ruff integrates with the tool nbQA to run over Jupyter Notebooks. After installing both ruff and nbqa, you can run Ruff on a notebook using a command like:

```
$ nbqa ruff Untitled.ipynb
```
Thought:Thought: Ruff uses nbQA to run over Jupyter Notebooks. Now I need to check if the president mentioned nbQA or Ruff in the state of the union address.
Action: State of Union QA System
Action Input: Did the president mention nbQA or Ruff in the most recent state of the union address?
Observation: No, the president did not mention nbQA or Ruff in the most recent State of the Union address.

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses the tool nbQA to run over Jupyter Notebooks. The president did not mention nbQA or Ruff in the most recent State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



In [27]:
from pathlib import Path

# Create a folder for our custom dataset.
Path("datasets").mkdir(exist_ok=True)

# Original sample data for testing our LegalTech agent.
legal_ai_text = """
LEGAL AI KNOWLEDGE BASE

Document review:
AI systems can assist lawyers by identifying clauses,
extracting relevant information, and comparing contracts.

Contract analysis:
AI-assisted contract analysis can identify payment terms,
termination clauses, confidentiality obligations, and
potentially inconsistent provisions.

Retrieval-Augmented Generation:
RAG combines document retrieval with language generation.
A retriever identifies relevant document passages, while
a language model uses those passages to generate answers.

Human oversight:
AI-generated legal analysis should be reviewed by qualified
professionals before being used in legal decisions.

Confidentiality:
Legal AI systems must handle client information carefully.
Access controls, secure storage, and appropriate data
processing safeguards are important considerations.
"""

# Save the text as a file that LangChain can load.
Path("datasets/legal_ai_knowledge.txt").write_text(
    legal_ai_text,
    encoding="utf-8"
)

print("Legal AI dataset created.")

Legal AI dataset created.


In [28]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_openai import OpenAIEmbeddings

# Load our custom Legal AI dataset.
loader = TextLoader("datasets/legal_ai_knowledge.txt")
documents = loader.load()

# Split the document into manageable passages.
splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separator="\n"
)

texts = splitter.split_documents(documents)

# Generate embeddings using OpenAI.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

# Store our document embeddings in Chroma.
legal_db = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="legal-ai-knowledge"
)

# Create a question-answering chain using our new knowledge base.
legal_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=legal_db.as_retriever()
)

print("Legal AI retrieval system ready.")

Legal AI retrieval system ready.


In [29]:
from langchain.agents import Tool, AgentType, initialize_agent

# Give the agent access to two specialized knowledge sources.
tools = [
    Tool(
        name="Legal AI Knowledge",
        func=legal_qa.run,
        description=(
            "Use this tool for questions about legal AI, "
            "contract review, legal document analysis, "
            "RAG, and human oversight."
        )
    ),
    Tool(
        name="Ruff Documentation",
        func=ruff.run,
        description=(
            "Use this tool for questions about Ruff, "
            "Python linting, and code quality."
        )
    )
]

# Initialize the agent with both tools.
custom_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    handle_parsing_errors=True
)

print("Custom agent ready.")

Custom agent ready.


In [30]:
# Test 1: Question requiring our Legal AI knowledge base.
result_1 = custom_agent.invoke(
    {"input": "What can AI identify during contract analysis?"}
)

print("TEST 1:", result_1["output"])


# Test 2: Question requiring the Ruff documentation.
result_2 = custom_agent.invoke(
    {"input": "Why might developers use Ruff instead of Flake8?"}
)

print("TEST 2:", result_2["output"])


# Test 3: Question involving both knowledge sources.
result_3 = custom_agent.invoke(
    {
        "input": (
            "How could AI assist contract review, "
            "and how could Ruff help maintain the Python code "
            "used to build that application?"
        )
    }
)

print("TEST 3:", result_3["output"])

TEST 1: During contract analysis, AI can identify key elements such as payment terms, termination clauses, confidentiality obligations, and potentially inconsistent or conflicting provisions within the contract. It can also highlight risks, obligations, and important dates or deadlines to assist in thorough contract review.
TEST 2: Developers might use Ruff instead of Flake8 because Ruff is significantly faster due to being implemented in Rust, offers a comprehensive set of built-in linting rules (including many Flake8 plugins), supports automatic fixing of many issues, works well alongside formatters like Black, supports modern Python versions, is easy to install, and can unify multiple linting and formatting tools into one. This makes Ruff a more efficient, powerful, and convenient choice compared to Flake8.
TEST 3: AI can assist contract review by automating the identification of key contract clauses (such as payment terms, termination, confidentiality), extracting relevant data poi